# Evaluating a support agent, layer by layer — with LangSmith

This notebook takes the **hierarchical CrewAI support crew** (manager + technical / billing / security specialists) and evaluates it across the five layers of an agentic stack:

| Layer | Question it answers | What we measure here | LangSmith feature used |
|---|---|---|---|
| **01 Application** | Is it fast, cheap and reliable? | E2E latency (p50/p95), time-to-first-LLM-response, tokens, cost / 1K tickets, errors | traces + **summary evaluators** |
| **02 Retrieval** | Did the knowledge base return the right documents? | Precision@k, Recall@k, MRR, NDCG@k, hit-rate | **experiment on the retriever alone** |
| **03 RAG** | Is the reply grounded in what was retrieved? | Faithfulness, Context Precision, Context Recall | **LLM-as-judge evaluators** |
| **04 Agent** | Did it route, act and finish the job well? | Task success, delegation accuracy, trajectory P/R, step efficiency | **`evaluate_existing`** on the same traces |
| **05 Memory** | Does it remember, update, abstain and isolate? | Recall, knowledge-update, abstention, isolation, memory tokens / query | **second dataset + kind-gated evaluators** |

**How the notebook is organised**

1. Setup and the agent (your crew, plus two small additions: a knowledge-base tool and a customer-memory step — the original crew had neither retrieval nor memory, so layers 02, 03 and 05 would have had nothing to measure).
2. Instrumentation: one `run_agent()` that returns *everything* an evaluator needs (answer, timings, tokens, tool steps, retrieved docs, memory facts).
3. Datasets in LangSmith.
4. One section per layer. We run the agent **once**, then attach each layer's scores to the same runs with `evaluate_existing` — so the layers are cheap to add and you can click from a low score straight into the trace.
5. A version comparison (A/B) and a CI-style quality gate.

> Every metric function is a plain Python function `(inputs, outputs, reference_outputs) -> score`, so you can unit-test it without any API call (see the offline sanity cell in Layer 02).

## 0 · Setup

In [ ]:
%pip install -q -U langsmith crewai google-genai python-dotenv pandas

In [ ]:
import os, re, json, math, time, threading, statistics
from IPython.display import display
from collections import OrderedDict
from dotenv import load_dotenv
load_dotenv()

# --- LangSmith -------------------------------------------------------------
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "support-agent-eval-demo")
# EU region? uncomment:  os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

for key in ["GOOGLE_API_KEY", "LANGSMITH_API_KEY"]:
    assert os.getenv(key), f"Set {key} in your environment or .env file"

# --- knobs you can change --------------------------------------------------
AGENT_MODEL = "gemini/gemini-3.1-flash-lite-preview"        # same as your crew
JUDGE_MODEL = os.getenv("JUDGE_MODEL", "gemini-3.1-flash-lite-preview")  # plain Gemini id; use a stronger model for judging if you can
PRICE_PER_1M = {"input": 0.10, "output": 0.40}              # USD per 1M tokens — PLACEHOLDER, put your real pricing here
K = 3                                                         # top-k for the KB search tool
REQUIRE_KB_USE = True   # tell specialists to look things up with search_kb. Set False to see what the crew does when only given the tool.

from langsmith import Client, traceable, evaluate, evaluate_existing
ls_client = Client()
print("LangSmith project:", os.environ["LANGSMITH_PROJECT"])

## 1 · The data the agent works with

### 1a · A small knowledge base (what the *retrieval* layer searches)
Twelve short policy / how-to documents, each with a stable `id`. Those ids are what we compare against the "gold" documents in the dataset. A tiny pure-Python **BM25** retriever keeps the demo dependency-free — swap in your vector store later and every metric below still works, because the evaluators only see ranked doc ids.

In [ ]:
KB = [
 {"id": "kb-login-01", "title": "Invalid credentials when logging in",
  "text": "If login fails with 'Invalid credentials', check caps lock, clear browser cache and cookies, and try a private window. After 5 failed attempts the account is locked for 15 minutes. If you use SSO, sign in through your identity provider instead of the password form."},
 {"id": "kb-reset-01", "title": "Password reset email not arriving",
  "text": "Reset emails normally arrive within 5 minutes from no-reply@acme.example. Check the spam folder and allow-list that sender. The reset link is valid for 30 minutes. You can request a new reset email after waiting 10 minutes."},
 {"id": "kb-billing-dup-01", "title": "Duplicate or double charges",
  "text": "Two charges on the same day are usually a pending authorization or a double click. A confirmed duplicate charge is refunded in full within 5-7 business days. Support can approve duplicate-charge refunds up to $100 without escalation."},
 {"id": "kb-refund-01", "title": "Refund policy",
  "text": "Monthly plans (Pro, Team) are refundable within 30 days of the charge. Annual plans are refundable within 14 days. Refunds go back to the original payment method; support cannot refund to a different card."},
 {"id": "kb-invoice-01", "title": "Understanding and fixing invoices",
  "text": "Invoices list the plan, seat count, tax and any proration. Download them from Billing > Invoices. Mid-cycle seat changes are prorated on the next invoice. If an invoice looks wrong, support can re-issue it after reviewing the seat history."},
 {"id": "kb-plans-01", "title": "Plans and pricing",
  "text": "Pro costs $20 per month. Team costs $45 per seat per month. Enterprise is custom priced. Downgrades take effect at the next billing date."},
 {"id": "kb-sec-compromise-01", "title": "Suspected account compromise",
  "text": "If you suspect your account was compromised: reset your password immediately, revoke all active sessions under Security > Active sessions, and enable two-factor authentication. Support will never ask for your password or one-time codes."},
 {"id": "kb-sec-geo-01", "title": "Login alert from a new country",
  "text": "A new-country login triggers an email alert. If it was not you, press 'Not me' in the email, which locks the session and starts the compromise checklist. Review Security > Active sessions for unfamiliar devices."},
 {"id": "kb-sec-2fa-01", "title": "Lost access to two-factor authentication",
  "text": "Use one of your backup codes to log in. If you have none, support can run identity verification using your account email and the last four digits of your latest invoice; this takes 24-48 hours. Never share a one-time code with support."},
 {"id": "kb-api-429-01", "title": "API rate limits and HTTP 429",
  "text": "HTTP 429 means the rate limit was exceeded. Pro allows 60 requests per minute, Team allows 600. Retry with exponential backoff and honour the Retry-After header."},
 {"id": "kb-crash-01", "title": "App crashes on launch",
  "text": "Update to the latest version, clear the app cache, and reinstall if needed. Send logs from Help > Diagnostics so engineering can investigate. Crashes after an update are tracked as incident reports."},
 {"id": "kb-sla-01", "title": "Support hours and response times",
  "text": "Support is available 24/7 for security issues and 9am-6pm weekdays for everything else. First response target is 4 business hours for Pro and 1 hour for Team."},
]
KB_BY_ID = {d["id"]: d for d in KB}

_STOP = set("a an the is are was were be to of and or in on for with it my i me you your we our this that at as by from not no do does did have has had can could would should but if so there their they them its".split())
def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9$]+", text.lower()) if t not in _STOP]

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = docs
        self.toks = [tokenize(d) for d in docs]
        self.k1, self.b = k1, b
        self.avgdl = (sum(len(t) for t in self.toks) / len(self.toks)) if self.toks else 0
        df = {}
        for t in self.toks:
            for w in set(t): df[w] = df.get(w, 0) + 1
        n = len(self.toks)
        self.idf = {w: math.log(1 + (n - f + .5) / (f + .5)) for w, f in df.items()}
    def search(self, query, k=3):
        q = tokenize(query)
        scores = []
        for i, t in enumerate(self.toks):
            tf = {}
            for w in t: tf[w] = tf.get(w, 0) + 1
            s = 0.0
            for w in q:
                if w in tf:
                    s += self.idf[w] * tf[w] * (self.k1 + 1) / (tf[w] + self.k1 * (1 - self.b + self.b * len(t) / (self.avgdl or 1)))
            scores.append((i, s))
        scores = [x for x in sorted(scores, key=lambda x: -x[1]) if x[1] > 0]
        return scores[:k]

KB_INDEX = BM25([d["title"] + ". " + d["text"] for d in KB])

@traceable(name="kb_retrieve", run_type="retriever")
def kb_retrieve(query: str, k: int = K):
    """Return ranked KB documents. run_type='retriever' makes LangSmith render them as documents."""
    hits = KB_INDEX.search(query, k)
    return [{"page_content": f'{KB[i]["title"]}. {KB[i]["text"]}',
             "metadata": {"doc_id": KB[i]["id"], "score": round(s, 3), "rank": r + 1}}
            for r, (i, s) in enumerate(hits)]

for h in kb_retrieve("I was charged twice for my Pro subscription", 3):
    print(h["metadata"], h["page_content"][:70], "...")

### 1b · A customer-memory store (what the *memory* layer manages)
Each customer has their **own namespace** of facts. Facts have a version and a `current` flag: writing a new value for the same attribute **supersedes** the old one (that is the *knowledge-update* behaviour), and recall only ever reads the caller's namespace (that is *isolation*). A `LEAKY_DEMO` switch deliberately breaks isolation so you can watch the metric catch it.

In [ ]:
LEAKY_DEMO = False   # flip to True later to see the isolation metric fail

MEMORY = {}          # customer_id -> list of fact dicts

def remember(customer_id, attribute, value, ts=None):
    """Write path: store a typed fact; supersede the previous value of the same attribute."""
    facts = MEMORY.setdefault(customer_id, [])
    for f in facts:
        if f["attribute"] == attribute and f["current"]:
            f["current"] = False
    facts.append({"attribute": attribute, "text": f"{attribute}: {value}", "current": True,
                  "version": 1 + sum(f["attribute"] == attribute for f in facts), "ts": ts or time.strftime("%Y-%m-%d")})

def seed_memory():
    MEMORY.clear()
    remember("C-1001", "Plan", "Pro, $20 per month", "2026-01-10")
    remember("C-1001", "Two-factor authentication", "enabled", "2026-02-02")
    remember("C-1001", "Language", "English", "2026-01-10")
    # C-1002 was on Team and later downgraded — a stale fact must NOT be served
    remember("C-1002", "Plan", "Team, 3 seats", "2026-05-01")
    remember("C-1002", "Plan", "Pro, $20 per month", "2026-08-01")          # supersedes the line above
    remember("C-1002", "Payment card", "Visa ending 4242", "2026-05-01")
    remember("C-1002", "Billing day", "3rd of each month", "2026-05-01")
    remember("C-1003", "Plan", "Team, 5 seats", "2026-03-15")
    remember("C-1003", "Account country", "India", "2026-03-15")
    remember("C-1003", "Two-factor authentication", "not enabled", "2026-03-15")
seed_memory()

@traceable(name="memory_recall", run_type="retriever")
def memory_recall(customer_id: str, query: str, k: int = 4):
    pool = [f for cid, fs in MEMORY.items() for f in fs if f["current"]] if LEAKY_DEMO else \
           [f for f in MEMORY.get(customer_id, []) if f["current"]]
    if LEAKY_DEMO:   # global search across ALL customers (the bug we want to detect)
        idx = BM25([f["text"] for f in pool]); hits = idx.search(query, k)
        pool = [pool[i] for i, _ in hits]
    else:
        pool = pool[:k]
    return [{"page_content": f["text"], "metadata": {"customer_id": customer_id, "version": f["version"], "ts": f["ts"]}} for f in pool]

print(memory_recall("C-1002", "which plan am I on"))

## 2 · The agent
The roles, goals and backstories are **exactly yours**. The changes: the specialists get a `search_kb` tool (so there is something to retrieve); step 2 of the task gets one added sentence telling them to use it and answer only from what it returns (`REQUIRE_KB_USE`, set it to `False` to see what the crew does when it is merely *given* the tool); the crew is built by a function, **fresh for every ticket** (CrewAI's token counters are cumulative per crew, so a reused crew would inflate every later ticket's tokens and cost); and `verbose` is off during evaluation.

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

# one recorder per run; evaluation runs sequentially (max_concurrency=0) so a global is safe here
class Recorder:
    def __init__(self): self.reset()
    def reset(self):
        self.lock = threading.Lock()
        self.kb_hits, self.steps, self.llm_done_ts = [], [], []
        self.t0 = None
REC = Recorder()

@tool("search_kb")
def search_kb(query: str) -> str:
    """Search the support knowledge base. Input: a short natural-language query. Returns the top matching articles with their ids."""
    hits = kb_retrieve(query, K)
    with REC.lock:
        REC.kb_hits.extend(hits)
    if not hits:
        return "No matching articles found."
    return "\n\n".join(f'[{h["metadata"]["doc_id"]}] {h["page_content"]}' for h in hits)

SPECIALIST_ROLES = ["Technical Support Engineer", "Billing Specialist", "Account Security Specialist"]

def build_crew(use_kb: bool = True, verbose: bool = False, require_kb: bool = False):
    llm = LLM(model=AGENT_MODEL, temperature=0.1, max_tokens=512, api_key=os.getenv("GOOGLE_API_KEY"))
    tools = [search_kb] if use_kb else []

    technical_agent = Agent(
        role="Technical Support Engineer",
        goal=("Resolve technical problems: login errors, crashes, bugs, API failures, "
              "performance issues, and integration problems. Give concrete, step-by-step fixes."),
        backstory=("You are a seasoned support engineer who has debugged thousands of production "
                   "issues. You ask for the minimum info needed, then give precise, actionable steps. "
                   "You never guess — if something is out of scope, you say so."),
        llm=llm, tools=tools, verbose=verbose, allow_delegation=False)

    billing_agent = Agent(
        role="Billing Specialist",
        goal=("Resolve billing and payment issues: charges, refunds, invoices, plan changes, "
              "subscription questions, and failed payments. Be precise about amounts and policy."),
        backstory=("You are a meticulous billing specialist. You explain charges clearly, follow "
                   "refund policy exactly, and never make commitments outside policy."),
        llm=llm, tools=tools, verbose=verbose, allow_delegation=False)

    security_agent = Agent(
        role="Account Security Specialist",
        goal=("Handle account-security issues: suspected compromise, password resets, 2FA, "
              "suspicious logins, and access recovery. Prioritise verifying identity safely."),
        backstory=("You are a calm, careful security specialist. You guide users through securing "
                   "their account without ever asking them to share passwords or one-time codes."),
        llm=llm, tools=tools, verbose=verbose, allow_delegation=False)

    support_manager = Agent(
        role="Customer Support Manager",
        goal=("Read the incoming ticket, decide which specialist(s) are actually needed, "
              "delegate ONLY to those, then synthesise a single clear reply to the customer."),
        backstory=("You are an experienced support lead. You triage accurately and do not waste "
                   "the team's time: a purely technical ticket goes to the technical engineer only; "
                   "a billing question goes to billing only; a ticket that spans domains goes to "
                   "multiple specialists. You combine their answers into one coherent response."),
        llm=llm, verbose=verbose, allow_delegation=True)

    resolve_ticket_task = Task(
        description=("A customer has submitted this support ticket:\n\n"
                     "-------------------------------------------------\n"
                     "{ticket}\n"
                     "-------------------------------------------------\n\n"
                     "Triage and resolve it:\n"
                     "1. Classify what it's actually about (technical, billing, account-security, "
                     "or a combination).\n"
                     "2. Delegate ONLY to the specialist(s) whose domain it touches."
                     + (" Tell each specialist to look up the relevant policy with the search_kb tool before answering, "
                        "and to answer only from what it returns (or say the knowledge base has no answer).\n" if (use_kb and require_kb) else "\n")
                     +
                     "3. Collect their resolution(s).\n"
                     "4. Write ONE final reply to the customer that resolves the issue clearly."),
        expected_output=("A single, customer-ready support reply that fully resolves the ticket, "
                         "reading as one coherent message."))

    return Crew(agents=[technical_agent, billing_agent, security_agent], tasks=[resolve_ticket_task],
                process=Process.hierarchical, manager_agent=support_manager, verbose=verbose)

print("crew builder ready")

## 3 · Instrumentation — turn one run into evaluable evidence
CrewAI publishes events (every tool call, every LLM call). We subscribe once and, for each run, collect:

* **steps** — every tool call, normalised to labels like `delegate:Billing Specialist` or `search_kb` (this is the *trajectory*),
* **LLM-call timestamps** — used for latency,
* **retrieved KB chunks** — what the agent actually saw,
* token usage from `result.token_usage`.

`run_agent()` is `@traceable`, so each call becomes one trace in LangSmith with the returned dict as its output. The evaluators in the later sections read that dict.

In [ ]:
from crewai.events import crewai_event_bus, ToolUsageStartedEvent, LLMCallCompletedEvent

def _label_step(tool_name, tool_args):
    name = (tool_name or "").strip()
    low = name.lower().replace("_", " ")          # newer CrewAI reports 'delegate_work_to_coworker', older 'Delegate work to coworker'
    if low.startswith("delegate work") or low.startswith("ask question"):
        blob = str(tool_args).lower()
        for role in SPECIALIST_ROLES:
            if role.lower() in blob:
                return f"delegate:{role}"
        return "delegate:unknown"
    return name

for _n, _e in (("_on_tool", ToolUsageStartedEvent), ("_on_llm", LLMCallCompletedEvent)):   # make the cell safe to re-run
    if _n in globals():
        try: crewai_event_bus.off(_e, globals()[_n])
        except Exception: pass

@crewai_event_bus.on(ToolUsageStartedEvent)
def _on_tool(source, event):
    with REC.lock:
        REC.steps.append(_label_step(getattr(event, "tool_name", ""), getattr(event, "tool_args", "")))

@crewai_event_bus.on(LLMCallCompletedEvent)
def _on_llm(source, event):
    with REC.lock:
        REC.llm_done_ts.append(event.timestamp.timestamp())

def approx_tokens(text: str) -> int:
    return max(1, round(len(text) / 4))        # rough rule of thumb; swap in tiktoken if you like

def cost_usd(prompt_tokens, completion_tokens):
    return (prompt_tokens * PRICE_PER_1M["input"] + completion_tokens * PRICE_PER_1M["output"]) / 1e6

import concurrent.futures, contextvars
def kickoff_in_thread(crew, inputs):
    """Jupyter already runs an asyncio event loop, and recent CrewAI versions refuse a synchronous kickoff() inside a running loop.
    Running it in a worker thread (which has no loop) avoids that, and copying the context keeps LangSmith tracing attached."""
    ctx = contextvars.copy_context()
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(ctx.run, crew.kickoff, inputs=inputs).result()

def get_crew(use_kb=True):
    # A FRESH crew per ticket: CrewAI's token_usage is cumulative per crew/LLM object, so reusing one crew makes
    # every ticket's tokens and cost include all earlier tickets (and the smoke test).
    return build_crew(use_kb=use_kb, require_kb=globals().get("REQUIRE_KB_USE", True))   # .get: still works if the setup cell was not re-run

@traceable(name="support_agent", run_type="chain")
def run_agent(ticket: str, customer_id: str = None, use_memory: bool = True, use_kb: bool = True) -> dict:
    REC.reset()
    memory_facts, memory_text = [], ""
    if use_memory and customer_id:
        mem = memory_recall(customer_id, ticket)
        memory_facts = [m["page_content"] for m in mem]
        if memory_facts:
            memory_text = f"\n\n[Customer context from memory — customer {customer_id}]\n" + "\n".join(f"- {f}" for f in memory_facts)

    t_start = time.time()
    error, answer, usage = None, "", None
    try:
        result = kickoff_in_thread(get_crew(use_kb), {"ticket": ticket + memory_text})
        answer, usage = result.raw, result.token_usage
    except Exception as e:                      # keep failures as data: error-rate is a Layer-01 metric
        error = f"{type(e).__name__}: {e}"
    latency = time.time() - t_start
    try: crewai_event_bus.flush()               # make sure handlers have drained
    except Exception: pass

    p_tok = getattr(usage, "prompt_tokens", 0) or 0
    c_tok = getattr(usage, "completion_tokens", 0) or 0
    first_llm = (min(REC.llm_done_ts) - t_start) if REC.llm_done_ts else None
    seen = OrderedDict()
    for h in REC.kb_hits: seen.setdefault(h["metadata"]["doc_id"], h["page_content"])
    return {
        "answer": answer, "error": error,
        "latency_s": round(latency, 3),
        "first_llm_response_s": round(first_llm, 3) if first_llm is not None else None,   # proxy for TTFT (kickoff() is not streamed)
        "llm_calls": len(REC.llm_done_ts),
        "prompt_tokens": p_tok, "completion_tokens": c_tok, "total_tokens": p_tok + c_tok,
        "cost_usd": cost_usd(p_tok, c_tok),
        "steps": list(REC.steps),
        "delegated_to": sorted({s.split(":", 1)[1] for s in REC.steps if s.startswith("delegate:") and s != "delegate:unknown"}),
        "retrieved_doc_ids": list(seen.keys()),
        "retrieved_context": list(seen.values()),
        "memory_facts": memory_facts,
        "memory_tokens": approx_tokens(memory_text) if memory_text else 0,
    }

# smoke test: one real ticket (costs a handful of LLM calls). Open the trace in LangSmith afterwards.
smoke = run_agent("I was charged twice for my Pro subscription this month — two $20 charges on the same day. I'd like one of them refunded.", "C-1002")
print(smoke["answer"][:500]); print({k: smoke[k] for k in ["latency_s","first_llm_response_s","llm_calls","total_tokens","cost_usd","steps","retrieved_doc_ids"]})

## 4 · Datasets in LangSmith
**Dataset A — `support-agent-e2e`** (layers 01–04): six tickets. For each one the *reference* says what a good run looks like:

* `expected_specialists` → delegation accuracy,
* `expected_steps` → trajectory,
* `relevance` → graded gold KB documents (gain 2 = essential, 1 = helpful) → retrieval metrics,
* `must_mention` → facts the reply has to contain → task success and context recall.

**Dataset B — `support-agent-memory`** (layer 05): five memory probes, each tagged with a `kind` (`recall`, `update`, `abstain`, `isolation`).

Set `RESET_DATASETS = True` after editing the examples to recreate them.

In [ ]:
RESET_DATASETS = False
TECH, BILL, SEC = "Technical Support Engineer", "Billing Specialist", "Account Security Specialist"

E2E_EXAMPLES = [
 {"inputs": {"ticket": "I keep getting an 'Invalid credentials' error when I log in, even though I'm sure my password is correct. The reset email never arrives either.", "customer_id": "C-1001"},
  "outputs": {"category": "technical", "expected_specialists": [TECH], "expected_steps": [f"delegate:{TECH}", "search_kb"],
              "relevance": {"kb-login-01": 2, "kb-reset-01": 2},
              "must_mention": ["check spam folder / allow-list the sender for the reset email", "reset link is valid for 30 minutes", "clear cache or try a private window; the account locks for 15 minutes after 5 failed attempts"]}},
 {"inputs": {"ticket": "I was charged twice for my Pro subscription this month — two $20 charges on the same day. I'd like one of them refunded.", "customer_id": "C-1002"},
  "outputs": {"category": "billing", "expected_specialists": [BILL], "expected_steps": [f"delegate:{BILL}", "search_kb"],
              "relevance": {"kb-billing-dup-01": 2, "kb-refund-01": 1},
              "must_mention": ["a confirmed duplicate charge is refunded in full", "refund takes 5-7 business days", "refund goes to the original payment method"]}},
 {"inputs": {"ticket": "I got an email saying someone logged into my account from another country. I didn't do that. Also my last invoice looks wrong — please check both.", "customer_id": "C-1003"},
  "outputs": {"category": "billing+security", "expected_specialists": [SEC, BILL], "expected_steps": [f"delegate:{SEC}", f"delegate:{BILL}", "search_kb"],
              "relevance": {"kb-sec-compromise-01": 2, "kb-sec-geo-01": 2, "kb-invoice-01": 2},
              "must_mention": ["reset the password and revoke active sessions", "enable two-factor authentication", "press 'Not me' / review active sessions", "download or review the invoice under Billing > Invoices; support can re-issue it"]}},
 {"inputs": {"ticket": "My API calls start failing with HTTP 429 after a few requests on the Pro plan. What should I do?", "customer_id": "C-1001"},
  "outputs": {"category": "technical", "expected_specialists": [TECH], "expected_steps": [f"delegate:{TECH}", "search_kb"],
              "relevance": {"kb-api-429-01": 2},
              "must_mention": ["429 means the rate limit was exceeded", "Pro allows 60 requests per minute", "retry with exponential backoff"]}},
 {"inputs": {"ticket": "I lost my phone and can't pass two-factor authentication to log in. How do I get back into my account?", "customer_id": "C-1003"},
  "outputs": {"category": "security", "expected_specialists": [SEC], "expected_steps": [f"delegate:{SEC}", "search_kb"],
              "relevance": {"kb-sec-2fa-01": 2},
              "must_mention": ["use a backup code if available", "otherwise identity verification via account email and last four digits of the latest invoice", "takes 24-48 hours", "never share one-time codes"]}},
 {"inputs": {"ticket": "The app crashes every time I open it since the last update, and I'm being billed for Team seats nobody uses anymore.", "customer_id": "C-1002"},
  "outputs": {"category": "technical+billing", "expected_specialists": [TECH, BILL], "expected_steps": [f"delegate:{TECH}", f"delegate:{BILL}", "search_kb"],
              "relevance": {"kb-crash-01": 2, "kb-invoice-01": 2, "kb-plans-01": 1},
              "must_mention": ["update, clear the cache or reinstall", "send logs from Help > Diagnostics", "seat changes are prorated on the next invoice / support can review seat history"]}},
]

MEMORY_EXAMPLES = [
 {"inputs": {"ticket": "Remind me which card I pay with and which day of the month I get billed.", "customer_id": "C-1002"},
  "outputs": {"kind": "recall", "expected_contains": ["4242", "3rd"]}},
 {"inputs": {"ticket": "What plan am I on right now and how much do I pay each month?", "customer_id": "C-1002"},
  "outputs": {"kind": "update", "expected_contains": ["Pro", "$20"], "stale_markers": ["3 seats"]}},
 {"inputs": {"ticket": "What phone number do you have on file for my account?", "customer_id": "C-1001"},
  "outputs": {"kind": "abstain"}},
 {"inputs": {"ticket": "Which payment card do you have on file for me?", "customer_id": "C-1003"},
  "outputs": {"kind": "isolation", "forbidden": ["4242", "Visa"]}},
 {"inputs": {"ticket": "How many seats do I have, and is two-factor authentication on?", "customer_id": "C-1003"},
  "outputs": {"kind": "recall", "expected_contains": ["5 seats"]}},
]

def ensure_dataset(name, description, examples):
    if ls_client.has_dataset(dataset_name=name):
        if not RESET_DATASETS:
            print(f"dataset '{name}' exists — reusing"); return name
        ls_client.delete_dataset(dataset_name=name)
    ls_client.create_dataset(name, description=description)
    ls_client.create_examples(dataset_name=name, examples=examples)
    print(f"created dataset '{name}' with {len(examples)} examples"); return name

DS_E2E = ensure_dataset("support-agent-e2e", "Tickets with routing, retrieval and fact references", E2E_EXAMPLES)
DS_MEM = ensure_dataset("support-agent-memory", "Memory probes: recall / update / abstain / isolation", MEMORY_EXAMPLES)

## 5 · Run the agent once, evaluate it five times
`evaluate()` runs the target over every example and stores one trace per example in a LangSmith **experiment**. We deliberately pass **no evaluators** here. Each layer below then calls `evaluate_existing()` on this same experiment to attach its scores — no second run of the (expensive) crew.

`max_concurrency=0` keeps runs sequential: the event recorder above is global, and sequential runs also give clean latency numbers.

In [ ]:
def target_e2e(inputs: dict) -> dict:
    return run_agent(**inputs)

exp = evaluate(target_e2e, data=DS_E2E, experiment_prefix="agent-baseline", max_concurrency=0,
               metadata={"version": "baseline", "use_kb": True, "use_memory": True})
EXP_BASELINE = exp.experiment_name
print("experiment:", EXP_BASELINE)

import pandas as pd
def scores(results, cols=None):
    """Mean of every feedback.* column (and, per example, a compact table)."""
    df = results.to_pandas()
    fb = [c for c in df.columns if c.startswith("feedback.")]
    if cols: fb = [c for c in fb if c.split("feedback.")[1] in cols]
    out = df[fb].apply(pd.to_numeric, errors="coerce").describe().loc[["count", "mean"]].T.round(3)
    out.index = [c.split("feedback.", 1)[1] for c in out.index]
    return out

# Layer 01 · Application
**Question:** is the system fast, cheap and reliable for a real user?

| Metric | Definition |
|---|---|
| `latency_e2e_s` | wall-clock seconds from request to final reply |
| `first_llm_response_s` | seconds until the first LLM call returns — a TTFT *proxy* (CrewAI's `kickoff()` isn't streamed; enable streaming if you need true TTFT) |
| `cost_usd` | `prompt_tokens × in_price + completion_tokens × out_price` |
| p50 / p95 latency | percentiles over the experiment; **p95 = smallest x with F(x) ≥ 0.95** (nearest-rank) |
| `cost_per_1k_tickets` | `1000 × total_spend / N_tickets` |
| `error_rate` | fraction of runs that raised |

Averages hide tails, so the summary evaluators report **p50 and p95** and you should alert on p95, not the mean.

In [ ]:
def latency_e2e_s(outputs):          return {"key": "latency_e2e_s", "score": outputs["latency_s"]}
def first_llm_response_s(outputs):   return {"key": "first_llm_response_s", "score": outputs.get("first_llm_response_s")}
def llm_calls(outputs):              return {"key": "llm_calls", "score": outputs["llm_calls"]}
def total_tokens(outputs):           return {"key": "total_tokens", "score": outputs["total_tokens"]}
def cost_per_ticket_usd(outputs):    return {"key": "cost_per_ticket_usd", "score": outputs["cost_usd"]}
def run_failed(outputs):             return {"key": "run_failed", "score": 1 if outputs.get("error") else 0, "comment": outputs.get("error")}

def percentile(values, p):
    """Nearest-rank percentile: smallest x such that at least p of the values are <= x."""
    xs = sorted(v for v in values if v is not None)
    if not xs: return None
    return xs[max(0, math.ceil(p * len(xs)) - 1)]

def app_summary(outputs: list[dict]):
    lat = [o["latency_s"] for o in outputs]
    n = len(outputs)
    spend = sum(o["cost_usd"] for o in outputs)
    return {"results": [
        {"key": "latency_p50_s", "score": percentile(lat, .50)},
        {"key": "latency_p95_s", "score": percentile(lat, .95)},
        {"key": "latency_mean_s", "score": sum(lat) / n},
        {"key": "cost_per_1k_tickets_usd", "score": 1000 * spend / n},
        {"key": "tickets_per_minute", "score": 60 * n / sum(lat) if sum(lat) else None},   # sequential run => 1 / mean latency
        {"key": "error_rate", "score": sum(1 for o in outputs if o.get("error")) / n},
    ]}

def local_summary(results) -> pd.Series:
    """Re-compute the summary metrics from the experiment's per-run outputs (public API only), so we can print/gate on them here.
    The same numbers are also uploaded to LangSmith by the summary evaluator."""
    outs = []
    for r in results.to_pandas().to_dict("records"):
        err = r.get("outputs.error")
        outs.append({"latency_s": r["outputs.latency_s"], "cost_usd": r["outputs.cost_usd"], "error": None if pd.isna(err) else err})
    return pd.Series({x["key"]: x["score"] for x in app_summary(outs)["results"]}).round(3)

L1 = evaluate_existing(EXP_BASELINE,
        evaluators=[latency_e2e_s, first_llm_response_s, llm_calls, total_tokens, cost_per_ticket_usd, run_failed],
        summary_evaluators=[app_summary])
display(scores(L1)); display(local_summary(L1).to_frame("summary"))

**Read it like this**
* *Layer 01 tells you the system is slow or expensive, not why.* Open the slowest trace in LangSmith: if time is in LLM calls, look at `llm_calls` (a chatty manager ↔ specialist loop is the usual culprit); if it is in tools, look at the retriever run.
* With a hierarchical crew, `llm_calls` and `total_tokens` explode when the manager delegates to specialists that were not needed — that is a Layer 04 problem showing up as a Layer 01 cost.
* In production, put `latency_e2e_s` p95 and `cost_per_ticket_usd` on a LangSmith dashboard and alert on them.

# Layer 02 · Retrieval
**Question:** did the knowledge base return the right documents, in the right order?

We evaluate the **retriever alone** — no LLM, no agent, milliseconds, free — against the graded gold `relevance` in dataset A. That isolates retrieval failures from everything above them.

For one query with `k` results, `rel_i ∈ {0,1}` is the relevance of rank `i`, `R` the number of gold documents:

* **Precision@k** = (relevant in top-k) / k   — how much noise is in the context
* **Recall@k** = (relevant in top-k) / R   — how much of what was needed arrived
* **MRR** = 1 / rank of the first relevant result   — how soon the first right answer appears
* **NDCG@k** = DCG@k / IDCG@k, with DCG = Σ gain_i / log2(i+1), gains 2/1 from the dataset   — rewards putting *essential* docs first
* **Hit-rate@k** = 1 if any relevant doc is in the top-k

In [ ]:
def _ranked_ids(retrieved):  # accepts list of doc ids or list of {"metadata": {"doc_id": ...}}
    return [r["metadata"]["doc_id"] if isinstance(r, dict) else r for r in retrieved]

def precision_at_k(ids, gold, k):  return sum(1 for d in ids[:k] if d in gold) / k
def recall_at_k(ids, gold, k):     return sum(1 for d in ids[:k] if d in gold) / len(gold) if gold else 0.0
def hit_rate_at_k(ids, gold, k):   return 1.0 if any(d in gold for d in ids[:k]) else 0.0
def reciprocal_rank(ids, gold):
    for i, d in enumerate(ids, 1):
        if d in gold: return 1 / i
    return 0.0
def ndcg_at_k(ids, gains, k):
    dcg = sum(gains.get(d, 0) / math.log2(i + 1) for i, d in enumerate(ids[:k], 1))
    ideal = sorted(gains.values(), reverse=True)[:k]
    idcg = sum(g / math.log2(i + 1) for i, g in enumerate(ideal, 1))
    return dcg / idcg if idcg else 0.0

# ---- offline sanity check (no API): the worked example from the slides ------
# 8 results, relevant at ranks 1,3,4,8 ; 5 gold docs overall (one never retrieved)
ids = ["d1", "x", "d2", "d3", "x", "x", "x", "d4"]
gold = {"d1", "d2", "d3", "d4", "d5"}
print("P@8", round(precision_at_k(ids, gold, 8), 2), "| R@8", round(recall_at_k(ids, gold, 8), 2),
      "| RR", round(reciprocal_rank(ids, gold), 2), "| NDCG@8", round(ndcg_at_k(ids, {g: 1 for g in gold}, 8), 2))

In [ ]:
def make_retrieval_evaluators(k):
    def _prep(outputs, reference_outputs):
        return _ranked_ids(outputs["retrieved"]), reference_outputs["relevance"]
    def p(outputs, reference_outputs):
        ids, rel = _prep(outputs, reference_outputs); return {"key": f"precision@{k}", "score": precision_at_k(ids, set(rel), k)}
    def r(outputs, reference_outputs):
        ids, rel = _prep(outputs, reference_outputs); return {"key": f"recall@{k}", "score": recall_at_k(ids, set(rel), k)}
    def m(outputs, reference_outputs):
        ids, rel = _prep(outputs, reference_outputs); return {"key": "mrr", "score": reciprocal_rank(ids, set(rel))}
    def n(outputs, reference_outputs):
        ids, rel = _prep(outputs, reference_outputs); return {"key": f"ndcg@{k}", "score": ndcg_at_k(ids, rel, k)}
    def h(outputs, reference_outputs):
        ids, rel = _prep(outputs, reference_outputs); return {"key": f"hit_rate@{k}", "score": hit_rate_at_k(ids, set(rel), k)}
    return [p, r, m, n, h]

def retriever_target(inputs: dict, k=K) -> dict:
    return {"retrieved": kb_retrieve(inputs["ticket"], k)}

rows = {}
for k in (1, 3, 5):                       # the cheapest experiment you will ever run: sweep k
    res = evaluate(lambda inp, k=k: retriever_target(inp, k), data=DS_E2E, evaluators=make_retrieval_evaluators(k),
                   experiment_prefix=f"retriever-k{k}", max_concurrency=0, metadata={"layer": 2, "k": k})
    s_ = scores(res)["mean"]
    s_.index = [re.sub(r"@\d+", "@k", i) for i in s_.index]        # precision@3 -> precision@k so the columns line up
    rows[k] = s_
pd.DataFrame(rows).rename_axis("metric").rename(columns=lambda k: f"k={k}").round(3)

**Read it like this**
* Precision falls and recall rises as `k` grows — that trade-off is the whole story of choosing `k`. More context also means more tokens (Layer 01) and more room for distraction (Layer 03).
* **MRR / NDCG** tell you about *ordering*. If recall@5 is fine but NDCG is low, add a re-ranker rather than raising `k`.
* A low `recall` for one specific ticket usually means the **ticket wording and the document wording don't overlap** (BM25) — look at that example in the experiment and fix with query rewriting or embeddings.
* Experiments `retriever-k1/3/5` appear side by side in LangSmith → *Compare* view.

### In-situ retrieval: what the agent really retrieved
The agent writes its *own* queries, so its retrieval can differ from the "ticket as query" test above. Same metrics, scored on the baseline traces:

In [ ]:
def agent_retrieval_recall(outputs, reference_outputs):
    return {"key": "agent_recall@all", "score": recall_at_k(outputs["retrieved_doc_ids"], set(reference_outputs["relevance"]), max(1, len(outputs["retrieved_doc_ids"])))}
def agent_retrieval_mrr(outputs, reference_outputs):
    return {"key": "agent_mrr", "score": reciprocal_rank(outputs["retrieved_doc_ids"], set(reference_outputs["relevance"]))}
def agent_searched(outputs):
    return {"key": "agent_called_search_kb", "score": 1 if "search_kb" in outputs["steps"] else 0}

L2b = evaluate_existing(EXP_BASELINE, evaluators=[agent_retrieval_recall, agent_retrieval_mrr, agent_searched])
scores(L2b)

# Layer 03 · RAG (is the reply grounded?)
**Question:** given what was retrieved, is the generated reply supported by it, and was the retrieved context actually useful?

Retrieval (Layer 02) can be perfect and the reply still invent things — or retrieval can miss a fact and the model fills the gap. Three LLM-as-judge metrics separate those cases. The judge sees only *context + answer* (never your reference answer for faithfulness), runs at temperature 0, and returns JSON.

| Metric | Computation |
|---|---|
| **Faithfulness** | split the reply into atomic claims; score = supported claims / all claims, where "supported" means *stated in the retrieved context or customer memory* |
| **Context precision** | judge marks each retrieved chunk relevant/irrelevant to the ticket; score = **average precision** over the ranks, so relevant chunks near the top score higher |
| **Context recall** | for each reference fact in `must_mention`, can it be found in the retrieved context? score = found / total |

*Low faithfulness + high context recall* = the model is making things up despite having the facts (prompt/model problem).
*Low context recall* = the facts never reached the model (Layer 02 problem).

In [ ]:
from google import genai
_gclient = None
def judge_json(prompt: str, retries: int = 2) -> dict:
    """LLM-as-judge call. Temperature 0, JSON out. Lazy client so the notebook imports offline."""
    global _gclient
    if _gclient is None: _gclient = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
    last = None
    for _ in range(retries + 1):
        try:
            resp = _gclient.models.generate_content(model=JUDGE_MODEL, contents=prompt,
                       config={"temperature": 0, "response_mime_type": "application/json"})
            txt = resp.text or ""
            try: return json.loads(txt)
            except Exception:
                m = re.search(r"\{.*\}", txt, re.S)
                if m: return json.loads(m.group(0))
        except Exception as e:
            last = e; time.sleep(1.5)
    raise RuntimeError(f"judge failed: {last}")

def _context(outputs, inputs=None):
    chunks = list(outputs.get("retrieved_context", [])) + list(outputs.get("memory_facts", []))
    return "\n".join(f"[{i+1}] {c}" for i, c in enumerate(chunks)) or "(empty)"

def average_precision(relevance_flags):
    """Rank-aware precision: mean of precision@i taken at each relevant rank i."""
    hits, total = 0, 0.0
    for i, rel in enumerate(relevance_flags, 1):
        if rel:
            hits += 1; total += hits / i
    return total / hits if hits else 0.0

def faithfulness(inputs, outputs):
    if outputs.get("error") or not outputs.get("answer"): return {"key": "faithfulness", "score": 0.0, "comment": "no answer"}
    j = judge_json(f"""You are a strict fact-checker.
CONTEXT (the only allowed source for policies, prices, limits and procedures):
{_context(outputs)}

CUSTOMER TICKET (details about the customer's own situation may be taken from here):
{inputs['ticket']}

ANSWER:
{outputs['answer']}

Split ANSWER into atomic factual claims about policies, prices, time frames, limits or procedures. Ignore greetings, apologies and offers of further help.
For each claim decide whether CONTEXT directly supports it.
Return JSON: {{"claims": [{{"claim": str, "supported": bool}}]}}""")
    claims = j.get("claims", [])
    if not claims: return {"key": "faithfulness", "score": 1.0, "comment": "no factual claims"}
    bad = [c["claim"] for c in claims if not c.get("supported")]
    return {"key": "faithfulness", "score": 1 - len(bad) / len(claims), "comment": ("unsupported: " + " | ".join(bad))[:500] if bad else "all claims supported"}

def context_precision(inputs, outputs):
    chunks = outputs.get("retrieved_context", [])
    if not chunks: return {"key": "context_precision", "score": 0.0, "comment": "nothing retrieved"}
    numbered = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(chunks))
    j = judge_json(f"""TICKET:
{inputs['ticket']}

RETRIEVED CHUNKS (in rank order):
{numbered}

For each chunk, say whether it contains information useful for resolving the ticket.
Return JSON: {{"relevant": [bool, ...]}} with exactly {len(chunks)} entries.""")
    flags = [bool(x) for x in j.get("relevant", [])][:len(chunks)]
    return {"key": "context_precision", "score": average_precision(flags), "comment": f"relevant flags by rank: {flags}"}

def context_recall(outputs, reference_outputs):
    facts = reference_outputs["must_mention"]
    j = judge_json(f"""CONTEXT:
{_context(outputs)}

REFERENCE FACTS:
{json.dumps(facts)}

For each reference fact, say whether CONTEXT states it or directly implies it.
Return JSON: {{"supported": [bool, ...]}} with exactly {len(facts)} entries.""")
    flags = [bool(x) for x in j.get("supported", [])][:len(facts)]
    flags += [False] * (len(facts) - len(flags))
    missing = [f for f, ok in zip(facts, flags) if not ok]
    return {"key": "context_recall", "score": sum(flags) / len(facts), "comment": ("missing from context: " + " | ".join(missing))[:500] if missing else "all facts present in context"}

# offline check of the rank-aware precision: relevant at ranks 1 and 3 -> (1/1 + 2/3)/2
print("AP check:", round(average_precision([True, False, True]), 3))

In [ ]:
L3 = evaluate_existing(EXP_BASELINE, evaluators=[faithfulness, context_precision, context_recall])
scores(L3)

**Read it like this**
* Click a low-faithfulness run → the evaluator's *comment* lists the unsupported claims; compare them with the retriever span to see whether the fact was missing from context or ignored.
* `context_precision` low with `precision@k` high means the *right ids* came back but the chunk text is too broad for the question — split long documents.
* Use a stronger model as `JUDGE_MODEL` than the one being judged when you can, and spot-check 10–20 judge verdicts by hand before trusting the trend.

# Layer 04 · Agent (routing, actions, outcome)
**Question:** did the manager choose the right specialists, take sensible steps, and actually resolve the ticket — without wasting calls?

The deck splits agent evaluation into **outcome** (did it finish?) and **trajectory** (how did it get there?). Both are scored from the `steps` we recorded:

| Metric | Computation |
|---|---|
| **Task success** | judge checks every `must_mention` fact is covered, the reply is a single coherent resolution, and nothing unsafe (asking for passwords / one-time codes). 1 only if all hold |
| **Fact coverage** | covered facts / required facts (a softer version of success) |
| **Delegation precision / recall** | specialists called vs `expected_specialists` — precision punishes unneeded delegation, recall punishes missed specialists |
| **Delegation exact-match** | 1 if the delegated set equals the expected set |
| **Trajectory precision / recall** | unique step labels taken vs `expected_steps` (order-insensitive) |
| **Step efficiency** | `min(1, expected_steps / actual_steps)` — 1.0 means no wasted tool calls |

In [ ]:
def task_success(inputs, outputs, reference_outputs):
    facts = reference_outputs["must_mention"]
    if outputs.get("error") or not outputs.get("answer"):
        return {"results": [{"key": "task_success", "score": 0, "comment": outputs.get("error") or "empty answer"},
                            {"key": "fact_coverage", "score": 0.0}]}
    j = judge_json(f"""You grade a customer-support reply.
TICKET:
{inputs['ticket']}

REPLY:
{outputs['answer']}

REQUIRED FACTS (each must be conveyed in the reply, wording may differ):
{json.dumps(facts)}

Return JSON:
{{"covered": [bool, ...]  /* exactly {len(facts)} entries, in order */,
  "single_coherent_reply": bool,
  "unsafe": bool  /* true if the reply asks the customer for a password or one-time code */}}""")
    cov = [bool(x) for x in j.get("covered", [])][:len(facts)]
    cov += [False] * (len(facts) - len(cov))
    ok = all(cov) and bool(j.get("single_coherent_reply", True)) and not j.get("unsafe", False)
    missing = [f for f, c in zip(facts, cov) if not c]
    return {"results": [
        {"key": "task_success", "score": 1 if ok else 0, "comment": ("missing: " + " | ".join(missing))[:400] if missing else ("unsafe" if j.get("unsafe") else "ok")},
        {"key": "fact_coverage", "score": sum(cov) / len(facts)}]}

def _set_pr(actual, expected):
    a, e = set(actual), set(expected)
    p = len(a & e) / len(a) if a else 0.0
    r = len(a & e) / len(e) if e else 1.0
    return p, r

def delegation_scores(outputs, reference_outputs):
    p, r = _set_pr(outputs["delegated_to"], reference_outputs["expected_specialists"])
    exact = 1 if set(outputs["delegated_to"]) == set(reference_outputs["expected_specialists"]) else 0
    return {"results": [{"key": "delegation_precision", "score": p}, {"key": "delegation_recall", "score": r},
                        {"key": "delegation_exact", "score": exact, "comment": f"called={outputs['delegated_to']} expected={reference_outputs['expected_specialists']}"}]}

def trajectory_scores(outputs, reference_outputs):
    p, r = _set_pr(outputs["steps"], reference_outputs["expected_steps"])
    return {"results": [{"key": "trajectory_precision", "score": p}, {"key": "trajectory_recall", "score": r,
                         "comment": f"steps={outputs['steps']}"}]}

def step_efficiency(outputs, reference_outputs):
    actual = max(1, len(outputs["steps"]))
    return {"key": "step_efficiency", "score": min(1.0, len(reference_outputs["expected_steps"]) / actual)}

# offline check: manager calls billing twice + search 3x for a 2-step ticket -> efficiency 2/5 = 0.4
print(step_efficiency({"steps": ["delegate:Billing Specialist"] * 2 + ["search_kb"] * 3}, {"expected_steps": ["delegate:Billing Specialist", "search_kb"]}))

In [ ]:
L4 = evaluate_existing(EXP_BASELINE, evaluators=[task_success, delegation_scores, trajectory_scores, step_efficiency])
scores(L4)

**Read it like this**
* **Success high, efficiency low** → it gets there by brute force (extra delegations / repeated searches). That is exactly the cost spike you saw in Layer 01.
* **Delegation precision low** → the manager over-delegates ("send everything to everyone"). Tighten the manager's backstory or add a routing example.
* **Delegation recall low** → a multi-domain ticket (e.g. security + invoice) was answered by only one specialist — the reply will fail `fact_coverage` for the missing half.
* **Success low but trajectory good** → right path, bad write-up: look at the manager's final synthesis step.
* The trace tree for each ticket shows the manager → specialist → tool structure, so you can jump from a metric to the exact step.

# Layer 05 · Memory
**Question:** does the agent remember what it should, update what changed, say "I don't know" when it doesn't, and never leak across customers?

Dataset B has one probe per behaviour. Each example carries a `kind`, and each evaluator **only scores the kinds it applies to** (returning `score=None` otherwise), so one evaluator list serves the whole dataset.

| Metric | Applies to | Pass condition |
|---|---|---|
| `memory_recall` | `recall` | fraction of `expected_contains` strings present in the reply |
| `knowledge_update` | `update` | current value present **and** stale value (`stale_markers`) absent |
| `abstention` | `abstain` | reply says it has no such record **and** invents no phone-like number |
| `isolation` | `isolation` | none of another customer's `forbidden` values appear in the reply **or** in the recalled memory facts |
| `memory_tokens_per_query` | all | tokens injected from memory (cost of remembering; compare with Layer 01 `total_tokens`) |

In [ ]:
ABSTAIN_PHRASES = ["don't have", "do not have", "no record", "not on file", "can't see", "cannot see", "unable to", "isn't on file", "not available", "no phone", "don't see", "do not see"]
PHONE_LIKE = re.compile(r"\+?\d[\d\s().-]{7,}\d")

def _na(key): return {"key": key, "score": None, "comment": "n/a for this kind"}

def memory_recall_score(outputs, reference_outputs):
    if reference_outputs.get("kind") != "recall": return _na("memory_recall")
    ans = outputs["answer"].lower(); exp = reference_outputs["expected_contains"]
    hit = [e for e in exp if e.lower() in ans]
    return {"key": "memory_recall", "score": len(hit) / len(exp), "comment": f"found {hit} of {exp}"}

def knowledge_update_score(outputs, reference_outputs):
    if reference_outputs.get("kind") != "update": return _na("knowledge_update")
    ans = outputs["answer"].lower()
    has_current = all(e.lower() in ans for e in reference_outputs["expected_contains"])
    has_stale = any(s.lower() in ans for s in reference_outputs.get("stale_markers", []))
    return {"key": "knowledge_update", "score": 1 if has_current and not has_stale else 0, "comment": f"current={has_current} stale_served={has_stale}"}

def abstention_score(outputs, reference_outputs):
    if reference_outputs.get("kind") != "abstain": return _na("abstention")
    ans = outputs["answer"]
    abstained = any(p in ans.lower() for p in ABSTAIN_PHRASES)
    fabricated = bool(PHONE_LIKE.search(ans))
    return {"key": "abstention", "score": 1 if abstained and not fabricated else 0, "comment": f"abstained={abstained} fabricated_number={fabricated}"}

def isolation_score(outputs, reference_outputs):
    if reference_outputs.get("kind") != "isolation": return _na("isolation")
    forb = [f.lower() for f in reference_outputs["forbidden"]]
    in_answer = any(f in outputs["answer"].lower() for f in forb)
    in_context = any(f in " ".join(outputs["memory_facts"]).lower() for f in forb)   # catches a leak even if the LLM didn't repeat it
    return {"key": "isolation", "score": 0 if (in_answer or in_context) else 1, "comment": f"leaked_in_answer={in_answer} leaked_in_recall={in_context}"}

def memory_tokens_per_query(outputs): return {"key": "memory_tokens_per_query", "score": outputs["memory_tokens"]}

# offline check: stale value served -> knowledge_update = 0
print(knowledge_update_score({"answer": "You are on Pro at $20, previously 3 seats on Team."}, {"kind": "update", "expected_contains": ["Pro", "$20"], "stale_markers": ["3 seats"]}))

In [ ]:
def target_memory(inputs: dict) -> dict:
    return run_agent(**inputs)

seed_memory()   # reset the store so the experiment is reproducible
exp_mem = evaluate(target_memory, data=DS_MEM, experiment_prefix="memory-baseline", max_concurrency=0,
                   evaluators=[memory_recall_score, knowledge_update_score, abstention_score, isolation_score, memory_tokens_per_query],
                   metadata={"layer": 5, "use_memory": True, "leaky": LEAKY_DEMO})
L5 = exp_mem
scores(L5)

**Read it like this**
* `knowledge_update` is the one most memory systems fail: the old fact stays in the store and gets recalled. Here the write path **supersedes** it (`remember()` flips `current=False`), so recall only sees the new value.
* `abstention` low → the agent fills gaps with plausible-sounding data. This is the most damaging failure for a support agent.
* `isolation` is a **gate, not a score**: anything below 1.0 is a privacy incident. We check *both* the reply and the recalled facts, so the leak is caught at retrieval even when the LLM happens to stay quiet.
* **Try the failure:** set `LEAKY_DEMO = True` in section 1b, re-run this cell, and watch `isolation` drop and the trace show another customer's card in the memory span.
* `memory_tokens_per_query` is the price of remembering — it should stay small as the store grows (retrieve top-k, don't dump everything).

# Putting it together
## A/B: does the knowledge-base tool earn its cost?
Same dataset, same evaluators, one change (`use_kb=False`). LangSmith stores both experiments so you can open *Compare* and diff individual traces.

In [ ]:
def target_no_kb(inputs: dict) -> dict:
    return run_agent(**{**inputs, "use_kb": False})

exp_nokb = evaluate(target_no_kb, data=DS_E2E, experiment_prefix="agent-no-kb", max_concurrency=0,
                    metadata={"version": "no-kb", "use_kb": False, "use_memory": True})
ALL_E2E = [latency_e2e_s, cost_per_ticket_usd, task_success, delegation_scores, step_efficiency, faithfulness]
r_no = evaluate_existing(exp_nokb.experiment_name, evaluators=ALL_E2E, summary_evaluators=[app_summary])
r_base = evaluate_existing(EXP_BASELINE, evaluators=ALL_E2E, summary_evaluators=[app_summary])
cmp = pd.concat({"baseline (with KB)": pd.concat([scores(r_base)["mean"], local_summary(r_base)]),
                 "no KB": pd.concat([scores(r_no)["mean"], local_summary(r_no)])}, axis=1)
cmp["delta"] = cmp["no KB"] - cmp["baseline (with KB)"]
cmp.round(3)

## CI-style quality gate
Turn the scores into pass/fail so a prompt, model or retriever change can't silently regress a layer. Run this cell (or the same logic in CI) after every experiment; fail the build when it asserts.

In [ ]:
THRESHOLDS = {   # metric -> (comparison, value)
    "recall@k": (">=", 0.70),
    "faithfulness": (">=", 0.85),
    "context_recall": (">=", 0.70),
    "task_success": (">=", 0.66),
    "delegation_exact": (">=", 0.66),
    "step_efficiency": (">=", 0.50),
    "latency_p95_s": ("<=", 60.0),
    "cost_per_1k_tickets_usd": ("<=", 5.00),
}
def gate(*result_sets, extra=()):
    merged = pd.concat([scores(r)["mean"] for r in result_sets] + [local_summary(result_sets[-1])] + list(extra))
    merged = merged[~merged.index.duplicated(keep="last")]
    failures = []
    for metric, (op, val) in THRESHOLDS.items():
        if metric not in merged.index: continue
        v = merged[metric]
        ok = v >= val if op == ">=" else v <= val
        print(("PASS" if ok else "FAIL"), f"{metric:28s} {v:8.3f}  (needs {op} {val})")
        if not ok: failures.append(metric)
    return failures

failed = gate(L3, L4, r_base, extra=[rows[3]])   # rows[3] = retriever metrics at k=3
# assert not failed, f"Quality gate failed: {failed}"      # uncomment in CI

## Where to go next
* **Online evaluation.** In LangSmith, add *automations* to the project so the cheap metrics (latency, cost, faithfulness judge on a sample) run on live traffic, and put p95 latency and cost per ticket on a dashboard with alerts.
* **Grow the datasets from production.** Turn low-scoring or thumbs-down traces into new dataset examples (*Add to dataset*) — the regression suite then reflects real failures.
* **Human labels.** Add annotation queues for faithfulness / task-success spot checks and measure judge-vs-human agreement.
* **Deeper traces.** CrewAI can also be traced span-by-span through OpenTelemetry (`openinference-instrumentation-crewai`) into LangSmith if you want every LLM call as its own span instead of the summary we record here.
* **Layer map.** Metric low in a layer? Check the layer *below* first: bad answers (04) are often bad grounding (03), which is often bad retrieval (02).